## 🌍 GDELT — Global Event Database Collection

### By:
MGO

### Date:
2026-03-23

### Description:

Collect global event data from GDELT (Global Database of Events, Language, and Tone)
for Colombian political content. GDELT monitors news media worldwide in 65+ languages
with 15-minute updates, providing tone scores that serve as an independent baseline
for our pysentimiento results.

> **Note:** ACLED conflict/protest event data has been moved to a dedicated notebook:
> `02_mgo_acled_collection_20260329.ipynb`. Run `make collect_acled` for that source.

**API:** GDELT DOC API (no authentication required — fully public)

**Data source priority:** P1 (Day 2)

**Output:** `data/01_raw/gdelt/gdelt_articles_YYYYMMDD.json`

## 📚 Import libraries

In [ ]:
from __future__ import annotations

import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Add src to path for importing project utilities
sys.path.insert(0, str(Path("../../src").resolve()))

from collectors.gdelt import query_gdelt
from utils.config import load_config

sns.set_theme(style="darkgrid")

## ⚙️ Configuration

Load GDELT filter settings from YAML config. GDELT uses theme codes and country
filters to narrow results to Colombian political coverage.

In [ ]:
from __future__ import annotations

# GDELT configuration
gdelt_config = load_config("../../conf/data_collection/gdelt.yml")
gdelt_filters = gdelt_config["filters"]
gdelt_settings = gdelt_config["settings"]

print("GDELT Configuration:")
print(f"  Country: {gdelt_filters['country_name']} ({gdelt_filters['country_code']})")
print(f"  Themes: {gdelt_filters['themes']}")
print(f"  Max records: {gdelt_settings['max_records']}")

## 💾 GDELT Collection

Query the GDELT DOC API for articles about Colombian politics. The API is free
and requires no authentication. It provides:
- Article URLs and titles
- Tone scores (-100 to +100) as an independent sentiment baseline
- Source domains and language metadata

GDELT tone scores give us a second opinion on sentiment that complements our
pysentimiento (Spanish-specific) model — useful for calibration and cross-validation.

In [ ]:
from __future__ import annotations

# Load election keywords for the GDELT query
keywords_config = load_config("../../conf/keywords/election.yml")
election_keywords = keywords_config["keywords"][:10]  # Top 10 for API query
print(f"Using keywords: {election_keywords}")

# Query GDELT (live HTTP request to public API -- no credentials needed)
try:
    df_gdelt = query_gdelt(
        country=gdelt_filters["country_name"],
        keywords=election_keywords,
        max_records=gdelt_settings["max_records"],
    )
    print(f"Collected {len(df_gdelt)} articles from GDELT")
except Exception as e:
    print(f"GDELT query failed: {e}")
    print("Creating empty DataFrame -- will retry later")
    df_gdelt = pd.DataFrame()

In [ ]:
from __future__ import annotations

if not df_gdelt.empty:
    print(f"Shape: {df_gdelt.shape}")
    print(f"Columns: {df_gdelt.columns.tolist()}")
    print("\nSample articles:")
    display(df_gdelt[["title", "source_domain", "tone"]].head(10))

    # Tone distribution
    print("\nTone statistics:")
    print(df_gdelt["tone"].describe())
    print(f"\nArticles with negative tone: {(df_gdelt['tone'] < 0).sum()}")
    print(f"Articles with positive tone: {(df_gdelt['tone'] > 0).sum()}")
else:
    print("No GDELT data collected -- check API availability")

In [ ]:
from __future__ import annotations

# Save GDELT data to raw layer
today = datetime.now().strftime("%Y%m%d")

if not df_gdelt.empty:
    gdelt_output_dir = Path("../../data/01_raw/gdelt")
    gdelt_output_dir.mkdir(parents=True, exist_ok=True)
    gdelt_output_path = gdelt_output_dir / f"gdelt_articles_{today}.json"
    df_gdelt.to_json(gdelt_output_path, orient="records", force_ascii=False, indent=2)
    print(f"Saved {len(df_gdelt)} GDELT articles to {gdelt_output_path}")
else:
    print("No GDELT data to save")

## 🔍 Exploratory Analysis

Inspect GDELT tone distribution and source breakdown to understand the
media landscape and calibrate against pysentimiento results.

In [ ]:
from __future__ import annotations

# GDELT tone timeline — summarize articles by day
if not df_gdelt.empty:
    df_gdelt_copy = df_gdelt.copy()
    df_gdelt_copy["date"] = pd.to_datetime(df_gdelt_copy["published"]).dt.strftime("%Y-%m-%d")
    timeline = df_gdelt_copy.groupby("date").agg(
        article_count=("title", "count"),
        avg_tone=("tone", "mean"),
    ).reset_index()

    print("GDELT daily article counts and average tone:")
    print(timeline.to_string(index=False))
else:
    print("No data available for analysis")

In [ ]:
from __future__ import annotations

# Visualize tone distribution and top source domains
if not df_gdelt.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Tone distribution histogram
    df_gdelt["tone"].hist(bins=30, ax=axes[0], color="steelblue", edgecolor="white")
    axes[0].axvline(0, color="red", linestyle="--", linewidth=1.5, label="Neutral (0)")
    axes[0].set_title("GDELT Tone Distribution\n(negative = adverse coverage)", fontsize=12)
    axes[0].set_xlabel("Tone Score")
    axes[0].set_ylabel("Article Count")
    axes[0].legend()

    # Top source domains
    top_sources = df_gdelt["source_domain"].value_counts().head(10)
    top_sources.plot(kind="barh", ax=axes[1], color="coral", edgecolor="white")
    axes[1].set_title("Top 10 Source Domains", fontsize=12)
    axes[1].set_xlabel("Article Count")
    axes[1].invert_yaxis()

    plt.suptitle("GDELT Colombia — Electoral Coverage", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No data to visualize")

## 📊 Collection Summary

In [ ]:
from __future__ import annotations

print("=" * 60)
print("GDELT COLLECTION SUMMARY")
print("=" * 60)

gdelt_count = len(df_gdelt) if not df_gdelt.empty else 0
print(f"GDELT articles collected: {gdelt_count}")

if not df_gdelt.empty:
    print(f"Date range: {df_gdelt['published'].min()} to {df_gdelt['published'].max()}")
    print(f"Unique source domains: {df_gdelt['source_domain'].nunique()}")
    print(f"Mean tone: {df_gdelt['tone'].mean():.2f}")
    print(f"Articles negative tone (<0): {(df_gdelt['tone'] < 0).sum()}")
    print(f"Articles positive tone (>0): {(df_gdelt['tone'] > 0).sum()}")

    print("\nTop 5 source domains:")
    for domain, count in df_gdelt["source_domain"].value_counts().head(5).items():
        print(f"  {domain:<40} {count}")

print("\nOutput file:")
print(f"  data/01_raw/gdelt/gdelt_articles_{today}.json")

## 💡 Notes & Next Steps

**GDELT's role in the analysis pipeline:**
- GDELT tone scores provide an **independent sentiment baseline** to cross-validate
  our Spanish-trained pysentimiento model. Where both agree, confidence is high;
  disagreements signal articles worth manual review.
- GDELT monitors 65+ languages, so it captures international reporting on Colombian
  elections that may not appear in our RSS (Spanish-language) feeds.

**Data quality observations:**
- GDELT coverage depends on online media indexing; rural events may be underreported
- Some GDELT articles may overlap with our RSS collection -- deduplication needed in the
  `02_intermediate` layer before combined analysis
- Tone is computed at article level, not paragraph level -- noisy for mixed-sentiment pieces

**Next steps:**
1. Deduplicate GDELT articles that overlap with RSS collection in `02_intermediate`
2. Compare GDELT tone trends with pysentimiento scores (notebook 3-analysis/01)
3. Use GDELT temporal coverage to fill gaps when RSS feeds are down

**Scheduling:**
- GDELT updates every 15 minutes; run `make collect_gdelt` daily or more frequently
- For ACLED conflict/protest events, run `make collect_acled` (notebook 02)

## 📖 References

- GDELT Project: https://gdeltproject.org
- GDELT DOC API: https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/
- CIVICUS Monitor Colombia: https://monitor.civicus.org/country/colombia/